# 📘 Guide 13: Streamlit and FastAPI

**Who it's for:** you'd like a head start on week 6, where GradeWise gets a teacher-facing app and an API. **Time:** about 30 minutes.

Two tools, two jobs:
- **Streamlit** turns a Python script into a web app for humans: upload scripts, click *Grade*, see report cards.
- **FastAPI** turns Python functions into a web API for other programs: a school's system could `POST` an answer and get a grade back as JSON.

### 🎯 You'll learn
1. A first Streamlit app, and how Streamlit re-runs your script
2. A first FastAPI endpoint, with Pydantic request/response models
3. Testing an API without starting a server
4. How the two fit together in GradeWise

## Part 1: Streamlit in 15 lines

Save this as `hello_grader.py` anywhere in the repo:

```python
import streamlit as st

st.title("📝 GradeWise: quick marker")
max_marks = st.slider("Max marks", 1, 10, 5)
answer = st.text_area("Student answer", "A stack is last-in, first-out.")

if st.button("Grade"):
    keywords = ["lifo", "last-in", "push", "pop"]
    hits = sum(k in answer.lower() for k in keywords)
    score = min(max_marks, hits * 2)
    st.metric("Score", f"{score} / {max_marks}")
    st.success("Nice work!") if score >= max_marks * 0.6 else st.warning("Needs review")
```

Run it from the terminal (same on Windows, Mac and Linux):

```bash
uv run python -m streamlit run hello_grader.py
```

Your browser opens at `http://localhost:8501`. Press `Ctrl + C` in the terminal to stop it.

💡 **The key idea:** every time you touch a widget, Streamlit re-runs the whole script from the top. Anything slow (loading a model, building a vector store) should be wrapped in `@st.cache_resource` so it happens once. Values you need to keep between runs go in `st.session_state`.

We can at least check Streamlit is installed:

In [ ]:
import streamlit

print("Streamlit", streamlit.__version__, "is installed ✅")

## Part 2: A first FastAPI endpoint

FastAPI uses type hints and Pydantic models (Guide 10) to validate requests and document your API automatically:

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(title="GradeWise mini API")


class GradeRequest(BaseModel):
    student: str
    answer: str
    max_marks: int = Field(default=5, gt=0)


class GradeResponse(BaseModel):
    student: str
    marks: int
    max_marks: int
    feedback: str


@app.get("/health")
def health() -> dict:
    return {"status": "ok"}


@app.post("/grade")
def grade(req: GradeRequest) -> GradeResponse:
    if not req.answer.strip():
        raise HTTPException(status_code=422, detail="Answer is empty")
    hits = sum(k in req.answer.lower() for k in ["lifo", "last-in", "push", "pop"])
    marks = min(req.max_marks, hits * 2)  # 💡 a stand-in for the real LLM grader
    return GradeResponse(student=req.student, marks=marks, max_marks=req.max_marks, feedback=f"Found {hits} key ideas")


print("Routes:", [r.path for r in app.routes][-2:])

## Part 3: Testing without a server

`TestClient` calls your app in-process, so it's fast, offline, and exactly how the course's tests check the week 6 API:

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

print(client.get("/health").json())

ok = client.post("/grade", json={"student": "Sam", "answer": "Push and pop, last-in first-out"})
print(ok.status_code, ok.json())

bad = client.post("/grade", json={"student": "Leo", "answer": "x", "max_marks": -1})
print(bad.status_code, bad.json()["detail"][0]["msg"])  # 💡 Pydantic rejected it for us

To run it as a real server, put the code in `mini_api.py` and:

```bash
uv run python -m uvicorn mini_api:app --reload
```

Then open `http://localhost:8000/docs`: FastAPI has generated interactive documentation where you can try every endpoint. `--reload` restarts the server whenever you save the file.

Calling it from another terminal:
- **Windows (PowerShell):** `Invoke-RestMethod -Method Post -Uri http://localhost:8000/grade -ContentType "application/json" -Body '{"student":"Sam","answer":"LIFO"}'`
- **Mac / Linux:** `curl -X POST http://localhost:8000/grade -H "Content-Type: application/json" -d '{"student":"Sam","answer":"LIFO"}'`

## Part 4: How they fit in GradeWise

```
 Asha (browser) ──▶ Streamlit app ──▶ gradewise package (graph, grader, RAG) ──▶ LLM
 School system  ──▶ FastAPI      ──▶      (same package)
```

Both are thin layers over the same `gradewise` package. Keep logic in the package and the app/API stay small and easy to test.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Long LLM jobs (grading a whole class) shouldn't block a web request. In week 6 you'll start the work in the background and let the UI poll for progress. Keep that in mind if your endpoint feels slow.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Try it</h2>
            <span style="color:#f59e0b;">Add a <code>POST /grade/batch</code> endpoint that accepts a list of <code>GradeRequest</code> and returns a list of <code>GradeResponse</code>, and test it with <code>TestClient</code>. Bonus: build the Streamlit app from Part 1 and run it.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• Streamlit documentation: <a href="https://docs.streamlit.io/">docs.streamlit.io/</a><br/>• FastAPI tutorial: <a href="https://fastapi.tiangolo.com/tutorial/">fastapi.tiangolo.com/tutorial/</a><br/>• FastAPI testing: <a href="https://fastapi.tiangolo.com/tutorial/testing/">fastapi.tiangolo.com/tutorial/testing/</a></span>
        </td>
    </tr>
</table>